# 14 — A1 MuJoCo adapter benchmark: 10 easy + 10 normal + 10 hard

## 背景
Notebook 13は4秒のequation-level proxyで、robot modelを動かす証拠ではない。
最終評価ではUnitree A1 MuJoCo plantを各scenario 20秒以上、転倒後も途中resetせず実行し、
定量metricとGIF playback時間を保存する。

## 目的
1. `src/legged_control_mujoco` のA1 adapterを30条件で実行した証拠を読む。
2. easy/normal/hardが各10件、simulation/GIFが各20秒以上か機械検証する。
3. pass/failだけでなく、失敗理由と物理metricを追って調整箇所へ戻る。

## 厳密な実装境界
- 上流正本: `external/legged_control/` commit `a7f381c0367e98e31c01336e678eef47e304d40d`。ROS1、OCS2 SQP-NMPC、
  Pinocchio/qpOASES WBC、Gazebo/Unitree I/Oの原実装。
- 実行対象: project所有 `src/legged_control_mujoco/adapter.py` と `models/a1.xml`。
- adapterはgait template、24D state/input contract、WBC task構造、hybrid torque式を対応させるが、
  **OCS2 SQPではない**。有限horizon policyを、瞬時friction-constrained force plannerと
  MuJoCo acceleration-level inverse dynamicsで置換する。
- したがって結果は「A1 MuJoCo adapter性能」であり、上流ROS1/OCS2 repository性能ではない。
- この経路は **Quadruped-PyMPCを一切使用しない**。

## 結論
保存されたmetricとGIFが揃ったscenarioだけを実行済みとみなす。閾値passはadapterについての
再現可能な判定であり、OCS2 SQPや実機A1の性能主張へ外挿しない。


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists():
        ROOT = candidate
        break

np.set_printoptions(precision=4, suppress=True)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True})
print("repository:", ROOT)

import csv
import json
from PIL import Image

SCENARIO_ROOT = ROOT / "notebook_legged" / "assets" / "scenarios"
JSON_PATH = SCENARIO_ROOT / "scenario_results.json"
CSV_PATH = SCENARIO_ROOT / "scenario_results.csv"


repository: /home/takuya/work/mpc_dog


## ASCIIデータフロー
```text
scenario(name,gait,command,friction,payload,push,seed)
  -> A1 MuJoCo model (q,v,contacts,M,b,J)
  -> instantaneous force planner
       min ||W(J^T f - (M qdd* + b))||² + lambda||f-f_nom||²
       fz>=0, |fx|<=mu*fz, |fy|<=mu*fz
  -> acceleration WBC
       stance/swing Cartesian qdd* + posture regularization
       M qdd + b - J^T f - S^T tau = 0
  -> hybrid torque
       tau_cmd=clip(tau+0(q*-q)+3(dq*-dq), +/-33.5)
  -> MuJoCo plant -- q,v,measured contact --> next control sample
  -> metrics + 20 s GIF -> JSON/CSV/gallery
```

## source / symbol / equation mapping
```cpp
// upstream: external/legged_control/legged_controllers/config/a1/gait.info
ModeSchedule::from_gait             // adapter.py: GAIT_TEMPLATES, mode_phase
// upstream: centroidal input first 12 entries
A1HeadlessAdapter::_optimize_contact_forces
  demand=(M*qdd+b)[0:6]             // floating-base wrench equation
  minimize ||W(J^T f-demand)||^2    // instantaneous; NOT OCS2 SQP
// upstream: WbcBase::formulateNoContactMotionTask/formulateSwingLegTask
A1HeadlessAdapter::_desired_qacc
  J*qdd = a_foot^* - Jdot*qdot      // stance/swing acceleration task
// upstream: WbcBase.cpp floating-base EoM and torque extraction
A1HeadlessAdapter::solve_wbc
  tau=(M*qdd+b-J^T*f)[actuated]      // then +/-33.5 N m
// upstream: LeggedController.cpp setCommand(...,0,3,tau)
adapter.py::hybrid_command           // ff + Kp error + Kd error
// project runner
scripts/run_legged_control_benchmark.py::run_scenario/write_aggregates
```


## 判定閾値
benchmark scriptの `Thresholds` が正本:

- simulation duration ≥ 20 s、GIF playback ≥ 20 s
- fallなし、minimum base height ≥ 0.18 m
- height RMSE ≤ 0.10 m、max |roll/pitch| ≤ 0.60 rad
- planar velocity RMSE ≤ 0.35 m/s、yaw-rate RMSE ≤ 0.60 rad/s
- max torque ≤ 33.5 N m、saturation fraction ≤ 0.10
- post-saturation dynamics residual ≤ 5.0
- planned/measured contact agreement ≥ 0.55

転倒検出自体はheight < 0.18 mまたは|roll/pitch| > 0.9 rad。判定姿勢閾値0.60 radの方が厳しい。


In [2]:
# --- Block 1: JSON/CSVを同時にloadし、30/10/10と名前集合を検証 ---
EXPECTED = [('easy', 'E01_stance_baseline'), ('easy', 'E02_stance_low'), ('easy', 'E03_stance_high'), ('easy', 'E04_walk_005'), ('easy', 'E05_walk_008'), ('easy', 'E06_walk_lateral'), ('easy', 'E07_stance_payload'), ('easy', 'E08_stance_gentle_push'), ('easy', 'E09_walk_turn'), ('easy', 'E10_walk_012'), ('normal', 'N01_walk_016'), ('normal', 'N02_walk_diagonal'), ('normal', 'N03_walk_turn'), ('normal', 'N04_dynamic_walk'), ('normal', 'N05_standing_trot'), ('normal', 'N06_trot'), ('normal', 'N07_walk_payload'), ('normal', 'N08_walk_push'), ('normal', 'N09_walk_mu045'), ('normal', 'N10_walk_low_turn_push'), ('hard', 'H01_walk_025'), ('hard', 'H02_walk_strafe'), ('hard', 'H03_walk_fast_turn'), ('hard', 'H04_trot_fast'), ('hard', 'H05_flying_trot'), ('hard', 'H06_pace'), ('hard', 'H07_low_friction'), ('hard', 'H08_heavy_payload'), ('hard', 'H09_strong_push'), ('hard', 'H10_compound')]
expected_names = [name for _, name in EXPECTED]
expected_levels = {level: sum(item[0] == level for item in EXPECTED)
                   for level in ("easy", "normal", "hard")}
assert len(EXPECTED) == 30 and expected_levels == {"easy": 10, "normal": 10, "hard": 10}

if not JSON_PATH.is_file() or not CSV_PATH.is_file():
    print("PENDING: run uv run python scripts/run_legged_control_benchmark.py --all")
    records, csv_rows = [], []
else:
    aggregate = json.loads(JSON_PATH.read_text(encoding="utf-8"))
    records = aggregate["results"]
    with CSV_PATH.open(newline="", encoding="utf-8") as handle:
        csv_rows = list(csv.DictReader(handle))
    json_names = [record["config"]["name"] for record in records]
    csv_names = [row["name"] for row in csv_rows]
    assert len(records) == len(csv_rows) == 30
    assert set(json_names) == set(csv_names) == set(expected_names)
    counts = {level: sum(r["config"]["difficulty"] == level for r in records)
              for level in ("easy", "normal", "hard")}
    assert counts == {"easy": 10, "normal": 10, "hard": 10}
    print("validated scenario counts:", counts)


validated scenario counts: {'easy': 10, 'normal': 10, 'hard': 10}


In [3]:
# --- Block 2: Pillowで全30 GIFのframe timingをdecodeして20秒以上を検証 ---
def gif_playback_seconds(path):
    total_ms = 0
    with Image.open(path) as image:
        frame_count = image.n_frames
        for index in range(frame_count):
            image.seek(index)
            total_ms += int(image.info.get("duration", 0))
    return frame_count, total_ms / 1000.0

gif_checks = []
for level, name in EXPECTED:
    path = SCENARIO_ROOT / "gifs" / f"{name}.gif"
    assert path.is_file(), f"missing GIF: {path}"
    frames, playback_s = gif_playback_seconds(path)
    assert frames > 0, f"empty GIF: {name}"
    assert playback_s + 1e-9 >= 20.0, f"{name}: GIF playback {playback_s:.3f} s < 20 s"
    gif_checks.append((level, name, frames, playback_s))
print(f"validated {len(gif_checks)} GIFs; minimum playback:",
      min(item[3] for item in gif_checks), "s")


validated 30 GIFs; minimum playback: 20.0 s


In [4]:
# --- Block 3: 保存metricを再表示し、失敗理由を隠さない ---
if records:
    summary = []
    for record in records:
        cfg, metric = record["config"], record["metrics"]
        summary.append({
            "name": cfg["name"], "difficulty": cfg["difficulty"],
            "passed": metric["passed"],
            "sim_s": metric["simulated_duration_s"],
            "gif_s": metric["gif_playback_duration_s"],
            "height_rmse_m": metric["height_error_rmse_m"],
            "velocity_rmse_mps": metric["velocity_tracking_rmse"]["planar_mps"],
            "max_rp_rad": metric["maximum_abs_roll_pitch_rad"],
            "max_tau_nm": metric["maximum_abs_torque_nm"],
            "dyn_residual": metric["maximum_dynamics_residual"],
            "contact_agreement": metric["planned_vs_measured_contact_agreement"],
            "failure_reasons": "; ".join(metric["failure_reasons"]) or "none",
        })
    import pandas as pd
    result_df = pd.DataFrame(summary).sort_values("name")
    display(result_df)
    display(result_df.groupby("difficulty").agg(
        scenarios=("name", "count"), passed=("passed", "sum"),
        mean_velocity_rmse=("velocity_rmse_mps", "mean"),
        worst_dynamics_residual=("dyn_residual", "max"),
    ))
else:
    print("PENDING: aggregate files are not complete yet")


,name,difficulty,passed,sim_s,gif_s,height_rmse_m,velocity_rmse_mps,max_rp_rad,max_tau_nm,dyn_residual,contact_agreement,failure_reasons
0,E01_stance_baseline,easy,True,20.0,20.0,0.001660,0.001765,0.002863,4.982440,0.000854,0.999500,none
1,E02_stance_low,easy,True,20.0,20.0,0.004407,0.005092,0.007001,6.457116,0.001525,0.999500,none
2,E03_stance_high,easy,True,20.0,20.0,0.001366,0.001223,0.002436,4.648692,0.000554,0.999500,none
3,E04_walk_005,easy,False,20.0,20.0,0.010655,0.182410,0.399830,33.500000,105.848700,0.667625,torque saturation fraction exceeds threshold; ...
4,E05_walk_008,easy,False,20.0,20.0,0.121545,0.371891,3.140771,33.500000,564.909449,0.546125,fall detected; base height below threshold; he...
5,E06_walk_lateral,easy,False,20.0,20.0,0.137871,0.416455,3.141419,33.500000,453.383257,0.512750,fall detected; base height below threshold; he...
6,E07_stance_payload,easy,True,20.0,20.0,0.001610,0.001717,0.003215,5.366698,0.000923,0.999500,none
7,E08_stance_gentle_push,easy,True,20.0,20.0,0.001662,0.007711,0.039548,7.023598,0.006408,0.999500,none
8,E09_walk_turn,easy,False,20.0,20.0,0.011505,0.114167,0.305804,33.500000,63.672441,0.676875,dynamics residual exceeds threshold
9,E10_walk_012,easy,False,20.0,20.0,0.105958,0.212197,3.140278,33.500000,314.266131,0.596375,fall detected; base height below threshold; he...


,scenarios,passed,mean_velocity_rmse,worst_dynamics_residual
difficulty,,,,
easy,10,5,0.131463,564.909449
hard,10,0,0.330295,733.932262
normal,10,0,0.342440,476.674242


## 生成時の測定結果
集約済み `30/30`、pass `5`、fail `25`。閾値判定はbenchmark scriptの保存値を表示する。

- `E01_stance_baseline` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E02_stance_low` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E03_stance_high` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E04_walk_005` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `E05_walk_008` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `E06_walk_lateral` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `E07_stance_payload` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E08_stance_gentle_push` (easy): **PASS**; sim 20.000 s; GIF 20.000 s; all thresholds met
- `E09_walk_turn` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; dynamics residual exceeds threshold
- `E10_walk_012` (easy): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold
- `N01_walk_016` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `N02_walk_diagonal` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; dynamics residual exceeds threshold
- `N03_walk_turn` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `N04_dynamic_walk` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold
- `N05_standing_trot` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `N06_trot` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `N07_walk_payload` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `N08_walk_push` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `N09_walk_mu045` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold
- `N10_walk_low_turn_push` (normal): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H01_walk_025` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H02_walk_strafe` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H03_walk_fast_turn` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H04_trot_fast` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H05_flying_trot` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H06_pace` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold
- `H07_low_friction` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H08_heavy_payload` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H09_strong_push` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold
- `H10_compound` (hard): **FAIL**; sim 20.000 s; GIF 20.000 s; fall detected, base height below threshold, height RMSE exceeds threshold, roll/pitch exceeds threshold, planar velocity RMSE exceeds threshold, yaw-rate RMSE exceeds threshold, torque saturation fraction exceeds threshold, dynamics residual exceeds threshold, contact agreement below threshold


## Easy 10


### `E01_stance_baseline`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E01_stance_baseline](assets/scenarios/gifs/E01_stance_baseline.gif)


### `E02_stance_low`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E02_stance_low](assets/scenarios/gifs/E02_stance_low.gif)


### `E03_stance_high`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E03_stance_high](assets/scenarios/gifs/E03_stance_high.gif)


### `E04_walk_005`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E04_walk_005](assets/scenarios/gifs/E04_walk_005.gif)


### `E05_walk_008`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E05_walk_008](assets/scenarios/gifs/E05_walk_008.gif)


### `E06_walk_lateral`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E06_walk_lateral](assets/scenarios/gifs/E06_walk_lateral.gif)


### `E07_stance_payload`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E07_stance_payload](assets/scenarios/gifs/E07_stance_payload.gif)


### `E08_stance_gentle_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E08_stance_gentle_push](assets/scenarios/gifs/E08_stance_gentle_push.gif)


### `E09_walk_turn`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E09_walk_turn](assets/scenarios/gifs/E09_walk_turn.gif)


### `E10_walk_012`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![E10_walk_012](assets/scenarios/gifs/E10_walk_012.gif)


## Normal 10


### `N01_walk_016`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N01_walk_016](assets/scenarios/gifs/N01_walk_016.gif)


### `N02_walk_diagonal`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N02_walk_diagonal](assets/scenarios/gifs/N02_walk_diagonal.gif)


### `N03_walk_turn`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N03_walk_turn](assets/scenarios/gifs/N03_walk_turn.gif)


### `N04_dynamic_walk`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N04_dynamic_walk](assets/scenarios/gifs/N04_dynamic_walk.gif)


### `N05_standing_trot`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N05_standing_trot](assets/scenarios/gifs/N05_standing_trot.gif)


### `N06_trot`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N06_trot](assets/scenarios/gifs/N06_trot.gif)


### `N07_walk_payload`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N07_walk_payload](assets/scenarios/gifs/N07_walk_payload.gif)


### `N08_walk_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N08_walk_push](assets/scenarios/gifs/N08_walk_push.gif)


### `N09_walk_mu045`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N09_walk_mu045](assets/scenarios/gifs/N09_walk_mu045.gif)


### `N10_walk_low_turn_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![N10_walk_low_turn_push](assets/scenarios/gifs/N10_walk_low_turn_push.gif)


## Hard 10


### `H01_walk_025`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H01_walk_025](assets/scenarios/gifs/H01_walk_025.gif)


### `H02_walk_strafe`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H02_walk_strafe](assets/scenarios/gifs/H02_walk_strafe.gif)


### `H03_walk_fast_turn`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H03_walk_fast_turn](assets/scenarios/gifs/H03_walk_fast_turn.gif)


### `H04_trot_fast`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H04_trot_fast](assets/scenarios/gifs/H04_trot_fast.gif)


### `H05_flying_trot`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H05_flying_trot](assets/scenarios/gifs/H05_flying_trot.gif)


### `H06_pace`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H06_pace](assets/scenarios/gifs/H06_pace.gif)


### `H07_low_friction`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H07_low_friction](assets/scenarios/gifs/H07_low_friction.gif)


### `H08_heavy_payload`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H08_heavy_payload](assets/scenarios/gifs/H08_heavy_payload.gif)


### `H09_strong_push`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H09_strong_push](assets/scenarios/gifs/H09_strong_push.gif)


### `H10_compound`
固有scenario定義・command・外乱・seedは
`scripts/run_legged_control_benchmark.py::SCENARIOS` を正本とする。

![H10_compound](assets/scenarios/gifs/H10_compound.gif)
